# 03 - M1: CNN + BiGRU over tonic-normalised pitch sequences

**Needs a GPU** (Runtime > Change runtime type > T4 GPU). Needs `splits/v1.csv` from notebook 01 -- if it
is not already under persistent storage, the next cell will prompt you to upload it from your computer.

M1 exists to pick up what B1 (the pitch-histogram baseline) deliberately throws away: melodic
*order* -- aroha/avaroha, short repeated phrases, glides. B1 is frozen at **95.1% recording-level
top-1 / 0.936 macro-F1** on the held-out test fold (see PLAN.md) -- that's the number this has to
beat to be worth the extra complexity. See CONCEPTS.md for why the input representation is built
the way it is.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/vishwesh-pujari/raaga-detector.git"
REPO_DIR = "/content/raaga-detector" if os.path.exists("/content") else "/kaggle/working/raaga-detector"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs are not always visible to a running kernel

# Google Drive is OPTIONAL. Mounting it only buys persistence of a few small files (a results
# table, a split file, some caches -- well under 100 MB) between sessions. Skip it if you are
# short on Drive space: everything below falls back to Colab's own local disk.
MOUNT_DRIVE = False  # flip to True if you have Drive space and want caches to persist
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed or was skipped, continuing on local disk:", e)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart runtime.")

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
import torch
from raaga import env
from raaga.data import catalog, hmd, saraga
from raaga.features import sequence_cache as SC
from raaga.eval import metrics
from raaga.models import sequence as M

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RAW, ROOT = env.raw_data_home(), env.root()
split_path = ROOT / "splits" / "v1.csv"
if not split_path.exists():
    from google.colab import files
    import shutil
    print("splits/v1.csv not found under", ROOT, "-- choose the v1.csv file from notebook 01")
    uploaded = files.upload()
    assert len(uploaded) == 1, f"expected exactly one file, got {list(uploaded)}"
    split_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(next(iter(uploaded)), split_path)
split = pd.read_csv(split_path)
INCLUDE_SARAGA = (split.dataset == "saraga").any()
N_CLASSES = split.raga.nunique()
CLASSES = sorted(split.raga.unique())
print(len(split), "recordings,", N_CLASSES, "ragas; Saraga:", INCLUDE_SARAGA, "| device:", DEVICE)

## 1. Features (sequence cache, built once)
Same download-skip behaviour as notebook 02: if the raw pitch files are already on local disk
(this session or persisted to Drive), nothing is re-downloaded. If the cache itself already exists
(e.g. from a previous run this session, or on Drive), this cell just loads it -- ~seconds.

In [ ]:
TONIC = "tonic"  # keep in sync with the B1 config being compared against
PARAMS = dict(chunk_s=30.0, hop_s=15.0, step_s=0.1)
cache = SC.cache_dir(ROOT, variant=f"pitch_{TONIC}", **PARAMS)
todo = [u for u in split.uid if not SC._file(cache, u).exists()]
print(len(todo), "recordings still to featurise")
if todo:
    hmd.download(RAW)
    if INCLUDE_SARAGA:
        saraga.download(RAW)
    paths = catalog.build_catalog(RAW, include_saraga=INCLUDE_SARAGA, tonic_kind=TONIC)[["uid", "pitch_path", "tonic_hz"]]
    SC.build(split[split.uid.isin(todo)].merge(paths, on="uid"), cache, **PARAMS)
data = SC.load(split, cache)
print(data.X.shape, "chunks x T x 3")

## 2. Cross-validation on folds 0-3 (test fold 4 untouched)
One default config to start (see CONCEPTS.md / PLAN.md for the architecture reasoning). Ablations
(bigger/smaller model, with/without class weighting, chunk length) are a deliberate follow-up, not
baked into this first pass -- same approach as the tonic-vs-tonicFine comparison for B1.

In [ ]:
fold_of = dict(zip(split.uid, split.fold))
fold_arr = np.array([fold_of[u] for u in data.uid])
y_idx_all = np.array([CLASSES.index(y) for y in data.y])

def run_fold(k, model_cfg, train_cfg, folds=(0, 1, 2, 3)):
    train = np.isin(fold_arr, [f for f in folds if f != k])
    val = fold_arr == k
    model = M.PitchSequenceModel(model_cfg)
    M.fit(model, data.X[train], y_idx_all[train], data.X[val], y_idx_all[val], train_cfg, verbose=False)
    proba = M.predict_proba(model, data.X[val], device=train_cfg.device)
    res = metrics.evaluate_predictions(proba, y_idx_all[val], data.uid[val], CLASSES)
    res["fold"] = k
    return res, model

MODEL_CFG = M.ModelConfig(n_classes=N_CLASSES, conv_channels=(32, 64), gru_hidden=128)
TRAIN_CFG = M.TrainConfig(epochs=30, batch_size=64, lr=1e-3, patience=6, device=DEVICE)
print(f"model params: {M.PitchSequenceModel(MODEL_CFG).n_params():,}")

cv_results = []
for k in (0, 1, 2, 3):
    res, _ = run_fold(k, MODEL_CFG, TRAIN_CFG)
    print(f"fold {k}: rec_top1={res['rec_top1']:.3f}  rec_macro_f1={res['rec_macro_f1']:.3f}  rec_ece={res['rec_ece']:.3f}")
    cv_results.append(res)

cv = pd.DataFrame([{k: v for k, v in r.items() if k not in ("confusion", "classes")} for r in cv_results])
display(cv)
print("mean rec_top1:", round(cv.rec_top1.mean(), 3), "+/-", round(cv.rec_top1.std(), 3))
print("mean rec_macro_f1:", round(cv.rec_macro_f1.mean(), 3))
print()
print("B1 frozen benchmark to beat: rec_top1=0.951 (CV mean 0.935), rec_macro_f1=0.936 (CV mean 0.923)")
print("chance level (rec top-1):", round(1 / N_CLASSES, 3))

In [ ]:
# where does it go wrong? confusion pooled over the CV folds
conf = sum(r["confusion"] for r in cv_results)
off = [(conf[i, j], CLASSES[i], CLASSES[j]) for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j and conf[i, j] > 0]
print("most confused (count, true, predicted):")
for n, t, p in sorted(off, reverse=True)[:12]:
    print(f"  {n}x  {t} -> {p}")
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(conf, cmap="Blues")
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=90, fontsize=7)
ax.set_yticks(range(len(CLASSES)), CLASSES, fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.tight_layout(); plt.show()

## 3. Final test (run ONCE, when M1 is frozen)
Trains on folds 0-3 combined and evaluates on the held-out fold 4. Set `RUN_TEST = True` deliberately,
same rule as B1: this fold gets touched once, after every other decision (config, ablations) is
already made from the CV numbers above -- see CONCEPTS.md's "held-out test set" section for why.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    tr = data.subset(split[split.fold != 4].uid)
    te = data.subset(split[split.fold == 4].uid)
    y_idx_tr = np.array([CLASSES.index(y) for y in tr.y])
    y_idx_te = np.array([CLASSES.index(y) for y in te.y])
    model = M.PitchSequenceModel(MODEL_CFG)
    M.fit(model, tr.X, y_idx_tr, te.X, y_idx_te, TRAIN_CFG, verbose=True)
    proba = M.predict_proba(model, te.X, device=DEVICE)
    out = metrics.evaluate_predictions(proba, y_idx_te, te.uid, CLASSES)
    summary = {k: v for k, v in out.items() if k not in ("confusion", "classes")}
    print(json.dumps(summary, indent=2))
    print()
    print("B1 (frozen):  rec_top1=0.951  rec_macro_f1=0.936")
    print(f"M1 (this run): rec_top1={summary['rec_top1']:.3f}  rec_macro_f1={summary['rec_macro_f1']:.3f}")
    (ROOT / "results").mkdir(exist_ok=True)
    (ROOT / "results" / "m1_test.json").write_text(json.dumps(summary, indent=2))
    torch.save(model.state_dict(), ROOT / "results" / "m1_weights.pt")